# Phase 2: Stochastic Modelling & Derivatives Pricing  
## Notebook 02_03 — Black-Scholes-Merton PDE and Analytical Pricing

### Research question

How does delta hedging transform a stochastic stock-price model into the Black-Scholes-Merton partial differential equation, and how can the analytical solution be validated computationally?

This notebook builds the continuous-time bridge from the previous binomial notebooks:

1. start from geometric Brownian motion;
2. apply Itô's lemma to an option value function;
3. construct a locally riskless delta-hedged portfolio;
4. derive the Black-Scholes-Merton PDE;
5. solve European call and put prices analytically;
6. validate put-call parity;
7. compute Greeks;
8. verify the PDE residual numerically;
9. compare analytical pricing with Monte Carlo risk-neutral pricing;
10. explain the assumptions and limitations of the model.

The key idea is:

> The Black-Scholes-Merton PDE appears because a continuously rebalanced delta hedge removes the Brownian risk locally, forcing the remaining riskless portfolio to earn the risk-free rate.

## 1. Model setup

Assume the stock price follows geometric Brownian motion under the risk-neutral measure $\mathbb Q$:

$$
dS_t = (r-q)S_tdt + \sigma S_tdW_t
$$
where:

- $S_t$ is the stock price;
- $r$ is the continuously compounded risk-free rate;
- $q$ is the continuous dividend yield;
- $\sigma$ is volatility;
- $W_t$ is Brownian motion under $\mathbb Q$.

Let $V(t,S)$ be the value of a derivative written on $S$.

The objective is to find $V(t,S)$ when the terminal payoff is known. For a European call:

$$
V(T,S) = \max(S-K,0)
$$
For a European put:

$$
V(T,S) = \max(K-S,0)
$$

## 2. Itô's lemma applied to the option value

Since $V(t,S_t)$ depends on both time and a stochastic process, ordinary calculus is not enough.

By Itô's lemma:

$$
\begin{aligned}
dV &= \frac{\partial V}{\partial t}dt \\
&\quad + \frac{\partial V}{\partial S}dS \\
&\quad + \frac{1}{2} \frac{\partial^2 V}{\partial S^2}(dS)^2
\end{aligned}
$$
Using:

$$
dS = (r-q)Sdt + \sigma SdW
$$
and the Itô rule:

$$
(dW)^2 = dt
$$
we get:

$$
\begin{aligned}
dV &= \left( V_t + (r-q)SV_S + \frac{1}{2}\sigma^2S^2V_{SS} \right)dt \\
&\quad + \sigma SV_SdW
\end{aligned}
$$
The option value is stochastic because of the $dW$ term.

## 3. Delta hedging and PDE derivation

Construct a portfolio:

$$
\Pi = V - \Delta S
$$
Choose:

$$
\Delta = V_S
$$
Then:

$$
d\Pi = dV - V_SdS
$$
Substitute $dV$ and $dS$. The Brownian term cancels, leaving:

$$
\begin{aligned}
d\Pi &= \left( V_t + \frac{1}{2}\sigma^2S^2V_{SS} \right)dt \\
&\quad - (r-q)SV_Sdt
\end{aligned}
$$
The portfolio $\Pi = V - V_SS$ is locally riskless, so it must earn the risk-free rate:

$$
d\Pi = r\Pi dt = r(V - SV_S)dt
$$
Equating terms and simplifying gives the Black-Scholes-Merton PDE:

$$
V_t + \frac{1}{2}\sigma^2S^2V_{SS} + (r-q)SV_S - rV = 0
$$
This is the central PDE solved by the Black-Scholes-Merton model.

## 4. Boundary and terminal conditions

A PDE needs conditions.

For a European call:

$$
V(T,S) = \max(S-K,0)
$$
As $S \to 0$:

$$
C(t,0) = 0
$$
As $S \to \infty$:

$$
C(t,S) \sim Se^{-q(T-t)} - Ke^{-r(T-t)}
$$
For a European put:

$$
V(T,S) = \max(K-S,0)
$$
As $S \to 0$:

$$
P(t,0) \sim Ke^{-r(T-t)}
$$
As $S \to \infty$:

$$
P(t,S) \to 0
$$

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from math import erf, exp, log, pi, sqrt
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)

In [ ]:
@dataclass(frozen=True)
class BSMConfig:
    """
    Black-Scholes-Merton configuration.
    """
    s0: float = 100.0
    strike: float = 100.0
    maturity: float = 1.0
    risk_free_rate: float = 0.05
    volatility: float = 0.20
    dividend_yield: float = 0.00

    def validate(self) -> None:
        if self.s0 <= 0:
            raise ValueError("s0 must be positive.")
        if self.strike <= 0:
            raise ValueError("strike must be positive.")
        if self.maturity <= 0:
            raise ValueError("maturity must be positive.")
        if self.volatility <= 0:
            raise ValueError("volatility must be positive.")


config = BSMConfig()
config.validate()

config

## 5. Standard normal helper functions

The analytical solution uses the standard normal cumulative distribution function:

$$
N(x) = \int_{-\infty}^{x}\frac{1}{\sqrt{2\pi}}e^{-z^2/2}dz
$$
and its density:

$$
\phi(x)=\frac{1}{\sqrt{2\pi}}e^{-x^2/2}
$$

In [ ]:
def normal_cdf(x: float | np.ndarray) -> float | np.ndarray:
    """
    Standard normal cumulative distribution function.
    """
    x_array = np.asarray(x)
    values = 0.5 * (1.0 + np.vectorize(erf)(x_array / sqrt(2.0)))

    if np.isscalar(x):
        return float(values)
    return values


def normal_pdf(x: float | np.ndarray) -> float | np.ndarray:
    """
    Standard normal probability density function.
    """
    x_array = np.asarray(x)
    values = np.exp(-0.5 * x_array ** 2) / sqrt(2.0 * pi)

    if np.isscalar(x):
        return float(values)
    return values

## 6. Analytical Black-Scholes-Merton formula

Let $\tau = T-t$ be time to maturity.

Define:

$$
\begin{aligned}
d_1 &= \frac{ \log(S/K) + (r-q+\frac{1}{2}\sigma^2)\tau } {\sigma\sqrt{\tau}}
\end{aligned}
$$
$$
d_2 = d_1 - \sigma\sqrt{\tau}
$$
The call price is:

$$
C = Se^{-q\tau}N(d_1) - Ke^{-r\tau}N(d_2)
$$
The put price is:

$$
P = Ke^{-r\tau}N(-d_2) - Se^{-q\tau}N(-d_1)
$$

In [ ]:
def bsm_d1_d2(
    stock_price: float | np.ndarray,
    strike: float,
    tau: float | np.ndarray,
    risk_free_rate: float,
    volatility: float,
    dividend_yield: float = 0.0
) -> tuple[np.ndarray, np.ndarray]:
    """
    Compute d1 and d2 for Black-Scholes-Merton.
    """
    s = np.asarray(stock_price, dtype=float)
    tau_arr = np.asarray(tau, dtype=float)

    tau_safe = np.maximum(tau_arr, 1e-12)

    d1 = (
        np.log(s / strike)
        + (risk_free_rate - dividend_yield + 0.5 * volatility ** 2) * tau_safe
    ) / (volatility * np.sqrt(tau_safe))

    d2 = d1 - volatility * np.sqrt(tau_safe)

    return d1, d2


def bsm_price(
    stock_price: float | np.ndarray,
    strike: float,
    tau: float | np.ndarray,
    risk_free_rate: float,
    volatility: float,
    option_type: str,
    dividend_yield: float = 0.0
) -> float | np.ndarray:
    """
    Black-Scholes-Merton European call or put price.
    """
    s = np.asarray(stock_price, dtype=float)
    tau_arr = np.asarray(tau, dtype=float)

    intrinsic_call = np.maximum(s - strike, 0.0)
    intrinsic_put = np.maximum(strike - s, 0.0)

    d1, d2 = bsm_d1_d2(
        stock_price=s,
        strike=strike,
        tau=tau_arr,
        risk_free_rate=risk_free_rate,
        volatility=volatility,
        dividend_yield=dividend_yield
    )

    call = (
        s * np.exp(-dividend_yield * tau_arr) * normal_cdf(d1)
        - strike * np.exp(-risk_free_rate * tau_arr) * normal_cdf(d2)
    )

    put = (
        strike * np.exp(-risk_free_rate * tau_arr) * normal_cdf(-d2)
        - s * np.exp(-dividend_yield * tau_arr) * normal_cdf(-d1)
    )

    call = np.where(tau_arr <= 1e-12, intrinsic_call, call)
    put = np.where(tau_arr <= 1e-12, intrinsic_put, put)

    if option_type == "call":
        result = call
    elif option_type == "put":
        result = put
    else:
        raise ValueError("option_type must be 'call' or 'put'.")

    if np.isscalar(stock_price) and np.isscalar(tau):
        return float(result)
    return result


call_price = bsm_price(
    stock_price=config.s0,
    strike=config.strike,
    tau=config.maturity,
    risk_free_rate=config.risk_free_rate,
    volatility=config.volatility,
    option_type="call",
    dividend_yield=config.dividend_yield
)

put_price = bsm_price(
    stock_price=config.s0,
    strike=config.strike,
    tau=config.maturity,
    risk_free_rate=config.risk_free_rate,
    volatility=config.volatility,
    option_type="put",
    dividend_yield=config.dividend_yield
)

pd.Series({
    "call_price": call_price,
    "put_price": put_price
})

## 7. Put-call parity validation

With continuous dividend yield, European put-call parity is:

$$
C - P = Se^{-q\tau} - Ke^{-r\tau}
$$
This is a fundamental no-arbitrage check.

In [ ]:
def put_call_parity_error(
    stock_price: float,
    strike: float,
    tau: float,
    risk_free_rate: float,
    dividend_yield: float,
    call_price: float,
    put_price: float
) -> float:
    """
    Return European put-call parity error.
    """
    rhs = (
        stock_price * exp(-dividend_yield * tau)
        - strike * exp(-risk_free_rate * tau)
    )
    return (call_price - put_price) - rhs


parity_error = put_call_parity_error(
    stock_price=config.s0,
    strike=config.strike,
    tau=config.maturity,
    risk_free_rate=config.risk_free_rate,
    dividend_yield=config.dividend_yield,
    call_price=call_price,
    put_price=put_price
)

pd.Series({
    "call_price": call_price,
    "put_price": put_price,
    "parity_error": parity_error
})

## 8. Analytical Greeks

The main Greeks are sensitivities of option value to inputs.

### Delta

$$
\Delta_C = e^{-q\tau}N(d_1)
$$
$$
\Delta_P = e^{-q\tau}(N(d_1)-1)
$$
### Gamma

$$
\Gamma = \frac{e^{-q\tau}\phi(d_1)}{S\sigma\sqrt{\tau}}
$$
Gamma is the same for calls and puts under the standard BSM model.

### Vega

$$
\nu = Se^{-q\tau}\phi(d_1)\sqrt{\tau}
$$
Vega is the sensitivity to volatility, not to one percentage point of volatility unless rescaled.

### Theta

Theta measures sensitivity to calendar time. The sign convention varies. In this notebook, theta is:

$$
\Theta = \frac{\partial V}{\partial t}
$$
where $t$ is calendar time, so increasing $t$ means moving closer to expiry.

In [ ]:
def bsm_greeks(
    stock_price: float,
    strike: float,
    tau: float,
    risk_free_rate: float,
    volatility: float,
    option_type: str,
    dividend_yield: float = 0.0
) -> pd.Series:
    """
    Analytical BSM Greeks for European call or put.
    """
    d1, d2 = bsm_d1_d2(
        stock_price=stock_price,
        strike=strike,
        tau=tau,
        risk_free_rate=risk_free_rate,
        volatility=volatility,
        dividend_yield=dividend_yield
    )

    d1 = float(d1)
    d2 = float(d2)

    pdf_d1 = normal_pdf(d1)
    exp_q = exp(-dividend_yield * tau)
    exp_r = exp(-risk_free_rate * tau)

    gamma = exp_q * pdf_d1 / (stock_price * volatility * sqrt(tau))
    vega = stock_price * exp_q * pdf_d1 * sqrt(tau)

    if option_type == "call":
        delta = exp_q * normal_cdf(d1)
        theta = (
            -stock_price * exp_q * pdf_d1 * volatility / (2 * sqrt(tau))
            - risk_free_rate * strike * exp_r * normal_cdf(d2)
            + dividend_yield * stock_price * exp_q * normal_cdf(d1)
        )
        rho = strike * tau * exp_r * normal_cdf(d2)
    elif option_type == "put":
        delta = exp_q * (normal_cdf(d1) - 1)
        theta = (
            -stock_price * exp_q * pdf_d1 * volatility / (2 * sqrt(tau))
            + risk_free_rate * strike * exp_r * normal_cdf(-d2)
            - dividend_yield * stock_price * exp_q * normal_cdf(-d1)
        )
        rho = -strike * tau * exp_r * normal_cdf(-d2)
    else:
        raise ValueError("option_type must be 'call' or 'put'.")

    return pd.Series({
        "d1": d1,
        "d2": d2,
        "delta": delta,
        "gamma": gamma,
        "vega": vega,
        "theta_calendar_time": theta,
        "rho": rho
    })


greeks_table = pd.DataFrame({
    "call": bsm_greeks(
        config.s0,
        config.strike,
        config.maturity,
        config.risk_free_rate,
        config.volatility,
        "call",
        config.dividend_yield
    ),
    "put": bsm_greeks(
        config.s0,
        config.strike,
        config.maturity,
        config.risk_free_rate,
        config.volatility,
        "put",
        config.dividend_yield
    )
})

greeks_table

## 9. Price surface over stock price and time to maturity

The analytical formula lets us evaluate $V(\tau,S)$ over a grid.

This surface is the PDE solution for a European option.

In [ ]:
stock_grid = np.linspace(40, 180, 120)
tau_grid = np.linspace(0.01, 1.0, 100)

S_mesh, tau_mesh = np.meshgrid(stock_grid, tau_grid)

call_surface = bsm_price(
    stock_price=S_mesh,
    strike=config.strike,
    tau=tau_mesh,
    risk_free_rate=config.risk_free_rate,
    volatility=config.volatility,
    option_type="call",
    dividend_yield=config.dividend_yield
)

put_surface = bsm_price(
    stock_price=S_mesh,
    strike=config.strike,
    tau=tau_mesh,
    risk_free_rate=config.risk_free_rate,
    volatility=config.volatility,
    option_type="put",
    dividend_yield=config.dividend_yield
)

call_surface.shape, put_surface.shape

In [ ]:
plt.figure(figsize=(10, 6))
plt.imshow(
    call_surface,
    aspect="auto",
    origin="lower",
    extent=[stock_grid.min(), stock_grid.max(), tau_grid.min(), tau_grid.max()]
)
plt.colorbar(label="Call price")
plt.title("Black-Scholes-Merton European Call Price Surface")
plt.xlabel("Stock price S")
plt.ylabel("Time to maturity tau")
plt.show()

## 10. Numerical PDE residual check

The BSM PDE in calendar time is:

$$
V_t + \frac{1}{2}\sigma^2S^2V_{SS} + (r-q)SV_S - rV = 0
$$
Since the analytical formula is usually written as a function of time to maturity:

$$
\tau = T-t
$$
we have:

$$
V_t = -V_\tau
$$
So the PDE becomes:

$$
-V_\tau + \frac{1}{2}\sigma^2S^2V_{SS} + (r-q)SV_S - rV = 0
$$
We approximate the derivatives with finite differences and verify that the residual is close to zero away from the payoff kink and boundaries.

In [ ]:
def compute_pde_residual_on_grid(
    stock_grid: np.ndarray,
    tau_grid: np.ndarray,
    values: np.ndarray,
    risk_free_rate: float,
    volatility: float,
    dividend_yield: float = 0.0
) -> pd.DataFrame:
    """
    Compute finite-difference BSM PDE residual in tau form:

    -V_tau + 0.5 sigma^2 S^2 V_SS + (r-q) S V_S - rV = 0.
    """
    dS = stock_grid[1] - stock_grid[0]
    dtau = tau_grid[1] - tau_grid[0]

    rows = []

    for i in range(1, len(tau_grid) - 1):
        for j in range(1, len(stock_grid) - 1):
            S = stock_grid[j]
            V = values[i, j]

            V_tau = (values[i + 1, j] - values[i - 1, j]) / (2 * dtau)
            V_S = (values[i, j + 1] - values[i, j - 1]) / (2 * dS)
            V_SS = (values[i, j + 1] - 2 * values[i, j] + values[i, j - 1]) / (dS ** 2)

            residual = (
                -V_tau
                + 0.5 * volatility ** 2 * S ** 2 * V_SS
                + (risk_free_rate - dividend_yield) * S * V_S
                - risk_free_rate * V
            )

            rows.append({
                "tau": tau_grid[i],
                "stock_price": S,
                "value": V,
                "pde_residual": residual,
                "abs_residual": abs(residual)
            })

    return pd.DataFrame(rows)


pde_residual_df = compute_pde_residual_on_grid(
    stock_grid=stock_grid,
    tau_grid=tau_grid,
    values=call_surface,
    risk_free_rate=config.risk_free_rate,
    volatility=config.volatility,
    dividend_yield=config.dividend_yield
)

pde_residual_df.describe()

In [ ]:
# Exclude regions close to the payoff kink and boundaries for a cleaner diagnostic.
residual_core = pde_residual_df[
    (pde_residual_df["stock_price"] > 60)
    & (pde_residual_df["stock_price"] < 160)
    & (np.abs(pde_residual_df["stock_price"] - config.strike) > 5)
    & (pde_residual_df["tau"] > 0.05)
].copy()

pd.Series({
    "mean_abs_residual_core": residual_core["abs_residual"].mean(),
    "median_abs_residual_core": residual_core["abs_residual"].median(),
    "max_abs_residual_core": residual_core["abs_residual"].max()
})

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(residual_core["pde_residual"], bins=80)
plt.title("Finite-Difference PDE Residuals for Analytical Call Surface")
plt.xlabel("PDE residual")
plt.ylabel("Count")
plt.show()

## 11. Monte Carlo validation under the risk-neutral measure

Under $\mathbb Q$, the terminal stock price is:

$$
S_T = S_0\exp\left((r-q-\frac{1}{2}\sigma^2)T + \sigma\sqrt{T}Z\right)
$$
where:

$$
Z \sim \mathcal N(0,1)
$$
A European payoff can be priced by:

$$
V_0 = e^{-rT}\mathbb E^{\mathbb Q}[g(S_T)]
$$
This should match the BSM analytical price up to Monte Carlo sampling error.

In [ ]:
def monte_carlo_bsm_price(
    config: BSMConfig,
    option_type: str,
    num_paths: int = 300_000,
    seed: int = 123
) -> dict:
    """
    Monte Carlo risk-neutral price for a European option under GBM.
    """
    local_rng = np.random.default_rng(seed)

    z = local_rng.standard_normal(num_paths)

    terminal_stock = config.s0 * np.exp(
        (config.risk_free_rate - config.dividend_yield - 0.5 * config.volatility ** 2) * config.maturity
        + config.volatility * sqrt(config.maturity) * z
    )

    if option_type == "call":
        payoff = np.maximum(terminal_stock - config.strike, 0.0)
    elif option_type == "put":
        payoff = np.maximum(config.strike - terminal_stock, 0.0)
    else:
        raise ValueError("option_type must be 'call' or 'put'.")

    discounted_payoff = np.exp(-config.risk_free_rate * config.maturity) * payoff

    price = float(discounted_payoff.mean())
    standard_error = float(discounted_payoff.std(ddof=1) / sqrt(num_paths))

    return {
        "option_type": option_type,
        "mc_price": price,
        "standard_error": standard_error,
        "ci_95_low": price - 1.96 * standard_error,
        "ci_95_high": price + 1.96 * standard_error,
        "num_paths": num_paths
    }


mc_call = monte_carlo_bsm_price(config, "call", num_paths=300_000, seed=123)
mc_put = monte_carlo_bsm_price(config, "put", num_paths=300_000, seed=456)

mc_comparison = pd.DataFrame([
    {
        **mc_call,
        "analytical_price": call_price,
        "mc_error": mc_call["mc_price"] - call_price
    },
    {
        **mc_put,
        "analytical_price": put_price,
        "mc_error": mc_put["mc_price"] - put_price
    }
])

mc_comparison

## 12. Delta hedging intuition from small shocks

The delta hedge removes the first-order exposure to small stock-price changes.

For a small change $\Delta S$:

$$
\Delta V \approx V_S\Delta S + \frac{1}{2}V_{SS}(\Delta S)^2
$$
If we hold:

$$
\Pi = V - V_SS
$$
then the first-order term cancels.

The remaining exposure is mostly second order, governed by gamma.

In [ ]:
def local_delta_hedge_shock_table(config: BSMConfig, option_type: str = "call") -> pd.DataFrame:
    """
    Compare option PnL and delta-hedged PnL for small spot shocks.
    """
    base_price = bsm_price(
        config.s0,
        config.strike,
        config.maturity,
        config.risk_free_rate,
        config.volatility,
        option_type,
        config.dividend_yield
    )

    delta = bsm_greeks(
        config.s0,
        config.strike,
        config.maturity,
        config.risk_free_rate,
        config.volatility,
        option_type,
        config.dividend_yield
    )["delta"]

    shocks = np.linspace(-0.05, 0.05, 41)
    rows = []

    for shock in shocks:
        shocked_s = config.s0 * (1 + shock)
        shocked_price = bsm_price(
            shocked_s,
            config.strike,
            config.maturity,
            config.risk_free_rate,
            config.volatility,
            option_type,
            config.dividend_yield
        )

        option_pnl = shocked_price - base_price
        hedge_pnl = option_pnl - delta * (shocked_s - config.s0)

        rows.append({
            "stock_return_shock": shock,
            "shocked_stock_price": shocked_s,
            "option_pnl": option_pnl,
            "delta_hedged_pnl": hedge_pnl
        })

    return pd.DataFrame(rows)


hedge_shock_df = local_delta_hedge_shock_table(config, "call")
hedge_shock_df.head()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(hedge_shock_df["stock_return_shock"], hedge_shock_df["option_pnl"], label="Unhedged option PnL")
plt.plot(hedge_shock_df["stock_return_shock"], hedge_shock_df["delta_hedged_pnl"], label="Delta-hedged PnL")
plt.axhline(0, linewidth=1)
plt.title("Local Delta Hedge Removes First-Order Stock Exposure")
plt.xlabel("Instantaneous stock return shock")
plt.ylabel("PnL")
plt.legend()
plt.show()

## 13. Boundary-condition checks

The analytical solution should satisfy the limiting boundary behaviour.

For calls, when $S$ is very large:

$$
C \approx Se^{-q\tau} - Ke^{-r\tau}
$$
For puts, when $S$ is very small:

$$
P \approx Ke^{-r\tau}
$$

In [ ]:
def boundary_condition_checks(config: BSMConfig) -> pd.Series:
    """
    Check approximate BSM boundary conditions at extreme stock prices.
    """
    low_s = 1e-6
    high_s = 1_000.0

    call_low = bsm_price(low_s, config.strike, config.maturity, config.risk_free_rate, config.volatility, "call", config.dividend_yield)
    call_high = bsm_price(high_s, config.strike, config.maturity, config.risk_free_rate, config.volatility, "call", config.dividend_yield)
    call_high_asymptotic = high_s * exp(-config.dividend_yield * config.maturity) - config.strike * exp(-config.risk_free_rate * config.maturity)

    put_low = bsm_price(low_s, config.strike, config.maturity, config.risk_free_rate, config.volatility, "put", config.dividend_yield)
    put_low_asymptotic = config.strike * exp(-config.risk_free_rate * config.maturity)
    put_high = bsm_price(high_s, config.strike, config.maturity, config.risk_free_rate, config.volatility, "put", config.dividend_yield)

    return pd.Series({
        "call_price_near_zero_stock": call_low,
        "call_high_stock_price": call_high,
        "call_high_stock_asymptotic": call_high_asymptotic,
        "call_high_boundary_error": call_high - call_high_asymptotic,
        "put_low_stock_price": put_low,
        "put_low_stock_asymptotic": put_low_asymptotic,
        "put_low_boundary_error": put_low - put_low_asymptotic,
        "put_price_high_stock": put_high
    })


boundary_checks = boundary_condition_checks(config)
boundary_checks

## 14. Sensitivity grids

Option prices are strongly affected by:

- spot price;
- strike;
- volatility;
- maturity;
- interest rates;
- dividend yield.

A simple volatility/moneyness grid helps reveal how convexity and optionality behave.

In [ ]:
def build_price_grid_over_moneyness_and_volatility(config: BSMConfig, option_type: str) -> pd.DataFrame:
    """
    Build option price grid over moneyness and volatility.
    """
    moneyness_grid = np.linspace(0.70, 1.30, 41)
    vol_grid = np.linspace(0.05, 0.80, 41)

    rows = []

    for m in moneyness_grid:
        for sigma in vol_grid:
            s = config.strike * m
            price = bsm_price(
                stock_price=s,
                strike=config.strike,
                tau=config.maturity,
                risk_free_rate=config.risk_free_rate,
                volatility=sigma,
                option_type=option_type,
                dividend_yield=config.dividend_yield
            )

            rows.append({
                "moneyness": m,
                "volatility": sigma,
                "stock_price": s,
                "option_type": option_type,
                "price": price
            })

    return pd.DataFrame(rows)


call_grid = build_price_grid_over_moneyness_and_volatility(config, "call")
call_grid.head()

In [ ]:
price_grid_pivot = call_grid.pivot(index="volatility", columns="moneyness", values="price")

plt.figure(figsize=(10, 6))
plt.imshow(
    price_grid_pivot.values,
    aspect="auto",
    origin="lower",
    extent=[
        call_grid["moneyness"].min(),
        call_grid["moneyness"].max(),
        call_grid["volatility"].min(),
        call_grid["volatility"].max()
    ]
)
plt.colorbar(label="Call price")
plt.title("BSM Call Price by Moneyness and Volatility")
plt.xlabel("Moneyness S/K")
plt.ylabel("Volatility")
plt.show()

## 15. Validation checklist

A BSM implementation should satisfy at least these checks:

1. call and put prices are non-negative;
2. put-call parity holds;
3. delta signs are sensible;
4. gamma is non-negative;
5. vega is non-negative;
6. PDE residual is close to zero away from the payoff kink;
7. Monte Carlo price agrees with analytical price within sampling error;
8. boundary conditions hold approximately;
9. price increases with volatility for vanilla options;
10. price approaches intrinsic value as $\tau \to 0$.

In [ ]:
def run_bsm_validation_checks(config: BSMConfig) -> pd.DataFrame:
    """
    Run basic validation checks for the BSM implementation.
    """
    c = bsm_price(config.s0, config.strike, config.maturity, config.risk_free_rate, config.volatility, "call", config.dividend_yield)
    p = bsm_price(config.s0, config.strike, config.maturity, config.risk_free_rate, config.volatility, "put", config.dividend_yield)
    call_g = bsm_greeks(config.s0, config.strike, config.maturity, config.risk_free_rate, config.volatility, "call", config.dividend_yield)
    put_g = bsm_greeks(config.s0, config.strike, config.maturity, config.risk_free_rate, config.volatility, "put", config.dividend_yield)

    vol_low = bsm_price(config.s0, config.strike, config.maturity, config.risk_free_rate, 0.10, "call", config.dividend_yield)
    vol_high = bsm_price(config.s0, config.strike, config.maturity, config.risk_free_rate, 0.40, "call", config.dividend_yield)

    near_expiry_call = bsm_price(config.s0, config.strike, 1e-8, config.risk_free_rate, config.volatility, "call", config.dividend_yield)
    intrinsic_call = max(config.s0 - config.strike, 0.0)

    rows = [
        {
            "check": "call_non_negative",
            "passed": c >= 0,
            "value": c
        },
        {
            "check": "put_non_negative",
            "passed": p >= 0,
            "value": p
        },
        {
            "check": "put_call_parity",
            "passed": abs(put_call_parity_error(config.s0, config.strike, config.maturity, config.risk_free_rate, config.dividend_yield, c, p)) < 1e-10,
            "value": put_call_parity_error(config.s0, config.strike, config.maturity, config.risk_free_rate, config.dividend_yield, c, p)
        },
        {
            "check": "call_delta_between_0_and_1",
            "passed": 0 <= call_g["delta"] <= 1,
            "value": call_g["delta"]
        },
        {
            "check": "put_delta_between_minus_1_and_0",
            "passed": -1 <= put_g["delta"] <= 0,
            "value": put_g["delta"]
        },
        {
            "check": "gamma_non_negative",
            "passed": call_g["gamma"] >= 0,
            "value": call_g["gamma"]
        },
        {
            "check": "vega_non_negative",
            "passed": call_g["vega"] >= 0,
            "value": call_g["vega"]
        },
        {
            "check": "call_price_increases_with_volatility",
            "passed": vol_high > vol_low,
            "value": vol_high - vol_low
        },
        {
            "check": "near_expiry_call_approaches_intrinsic",
            "passed": abs(near_expiry_call - intrinsic_call) < 1e-6,
            "value": near_expiry_call - intrinsic_call
        },
        {
            "check": "core_pde_residual_small",
            "passed": residual_core["abs_residual"].median() < 1e-2,
            "value": residual_core["abs_residual"].median()
        }
    ]

    return pd.DataFrame(rows)


validation_checks = run_bsm_validation_checks(config)
validation_checks

## 16. Limitations

The Black-Scholes-Merton model is mathematically elegant, but its assumptions are restrictive.

### 16.1 Constant volatility

The model assumes one constant volatility parameter.

Real option markets display volatility smiles, skews, and term structures.

### 16.2 Continuous hedging

The derivation assumes continuous rebalancing.

Real hedging is discrete and incurs transaction costs, bid-ask spreads, latency, and market impact.

### 16.3 Lognormal diffusion

The model assumes continuous stock paths with normally distributed log returns.

Real markets have jumps, heavy tails, stochastic volatility, and rough volatility.

### 16.4 Constant rates and dividend yield

The model assumes deterministic constant $r$ and $q$.

Rates, funding costs, borrow fees, and dividends may be stochastic or discrete.

### 16.5 Frictionless markets

The model assumes unlimited borrowing/lending, no short-sale constraints, no taxes, no margin frictions, and perfect liquidity.

### 16.6 European exercise only

The closed-form formula applies to European vanilla options.

American, Bermudan, barrier, Asian, lookback, and other path-dependent options require numerical methods or alternative analytical tools.

## 17. Research frontier and current directions

The BSM PDE is classical, but it remains the starting point for many active research directions.

### 17.1 Local volatility and implied-volatility surfaces

The BSM model uses constant volatility.

Local volatility models replace $\sigma$ with $\sigma(t,S)$ and attempt to fit the full implied-volatility surface.

A future notebook could derive the Dupire local-volatility equation and calibrate it to synthetic option prices.

### 17.2 Stochastic volatility and Heston-type PDEs

In stochastic volatility models, volatility becomes a state variable.

This turns the one-dimensional BSM PDE into a higher-dimensional PDE.

A future notebook could compare Black-Scholes and Heston prices across strikes and maturities.

### 17.3 Finite-difference pricing

Many derivatives have no closed-form formula.

Finite-difference methods solve the pricing PDE directly on a grid and are especially useful for American options and barrier options.

A future notebook could implement explicit, implicit, and Crank-Nicolson schemes.

### 17.4 Deep BSDE methods for high-dimensional PDEs

High-dimensional derivative pricing problems suffer from the curse of dimensionality.

Deep BSDE methods reformulate certain PDEs as backward stochastic differential equations and use neural networks to approximate high-dimensional solutions.

A future notebook could introduce a toy high-dimensional basket option pricing problem.

### 17.5 Physics-informed neural networks

Physics-informed neural networks impose PDE residuals inside the loss function.

For option pricing, a PINN can be trained to satisfy the BSM PDE, terminal condition, and boundary conditions.

A future notebook could compare an analytical BSM solution with a small PINN solution.

### 17.6 Rough volatility and fractional PDEs

Empirical volatility often behaves more irregularly than classical diffusion models assume.

Rough volatility and fractional PDE approaches attempt to capture memory and non-Markovian behaviour.

A future notebook could compare BSM with rough Bergomi-style simulations or fractional Black-Scholes PDE approximations.

### 17.7 XVA and funding adjustments

Real derivative desks price funding, counterparty credit, collateral, and capital effects.

These lead to nonlinear PDEs or BSDEs that extend far beyond the classical BSM framework.

A future notebook could conceptually compare clean BSM price with funding-adjusted valuation.

## 18. Suggested follow-up notebooks

This notebook naturally leads to:

1. **02_04_implied_volatility_root_finding**  
   Inverting BSM prices to recover implied volatility.

2. **02_08_dynamic_delta_hedging_simulation**  
   Testing BSM delta hedging under discrete rebalancing.

3. **02_10_finite_difference_pde_pricer**  
   Solving the BSM PDE numerically on a grid.

4. **02_11_heston_model_calibration**  
   Moving from constant volatility to stochastic volatility.

5. **02_14_local_volatility_dupire_surface**  
   Inferring local volatility from an implied-volatility surface.

6. **03_or_05_arbitrage_free_ml_constraints**  
   Testing whether learned option surfaces violate no-arbitrage.

7. **06_deep_hedging_with_transaction_costs**  
   Learning hedges under transaction costs and market frictions.

## 19. Saving outputs

The notebook saves:

1. analytical call and put prices;
2. Greeks;
3. Monte Carlo comparison;
4. PDE residual diagnostics;
5. validation checks;
6. call price surface;
7. manifest with model assumptions.

In [ ]:
output_dir = Path("data/processed/black_scholes_merton_pde_v1")
output_dir.mkdir(parents=True, exist_ok=True)

prices_path = output_dir / "bsm_prices.csv"
greeks_path = output_dir / "bsm_greeks.csv"
mc_path = output_dir / "monte_carlo_comparison.csv"
residual_path = output_dir / "pde_residuals_core.csv"
validation_path = output_dir / "validation_checks.csv"
surface_path = output_dir / "call_price_surface.csv"
manifest_path = output_dir / "manifest.json"

prices_df = pd.DataFrame([
    {
        "option_type": "call",
        "price": call_price,
        **asdict(config)
    },
    {
        "option_type": "put",
        "price": put_price,
        **asdict(config)
    }
])

surface_df = pd.DataFrame({
    "stock_price": S_mesh.ravel(),
    "tau": tau_mesh.ravel(),
    "call_price": call_surface.ravel(),
    "put_price": put_surface.ravel()
})

prices_df.to_csv(prices_path, index=False)
greeks_table.to_csv(greeks_path)
mc_comparison.to_csv(mc_path, index=False)
residual_core.to_csv(residual_path, index=False)
validation_checks.to_csv(validation_path, index=False)
surface_df.to_csv(surface_path, index=False)

manifest = {
    "dataset_name": "black_scholes_merton_pde_outputs",
    "schema_version": "black_scholes_merton_pde_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "call_price": float(call_price),
    "put_price": float(put_price),
    "parity_error": float(parity_error),
    "median_abs_pde_residual_core": float(residual_core["abs_residual"].median()),
    "notes": [
        "BSM PDE assumes constant volatility, continuous hedging, and frictionless markets.",
        "PDE residual is computed numerically on the analytical solution surface.",
        "Monte Carlo validation uses risk-neutral GBM terminal sampling.",
        "This notebook does not implement a finite-difference solver; that is reserved for 02_10."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

prices_path, greeks_path, mc_path, residual_path, validation_path, surface_path, manifest_path

## 20. Summary

This notebook derived and implemented the Black-Scholes-Merton PDE.

It showed that:

1. geometric Brownian motion plus Itô's lemma gives stochastic option dynamics;
2. delta hedging removes the Brownian term locally;
3. the resulting locally riskless portfolio must earn the risk-free rate;
4. this produces the PDE:

$$
V_t + \frac{1}{2}\sigma^2S^2V_{SS} + (r-q)SV_S - rV = 0
$$
5. European call and put prices have closed-form analytical solutions;
6. put-call parity validates the implementation;
7. Greeks measure sensitivities and connect to hedging;
8. the analytical surface approximately satisfies the PDE residual numerically;
9. Monte Carlo risk-neutral pricing agrees with the analytical formula within sampling error.

The key mathematical takeaway is:

> The BSM formula is not just a formula. It is the solution to a no-arbitrage PDE created by dynamic replication.

The key financial takeaway is:

> Continuous-time pricing is elegant, but its assumptions are fragile. Real desks must account for volatility smiles, jumps, discrete hedging, transaction costs, funding, and model risk.

## 21. Further reading

### Classical foundations

- Black, F. and Scholes, M. "The Pricing of Options and Corporate Liabilities."
- Merton, R. C. "Theory of Rational Option Pricing."
- Shreve, S. E. *Stochastic Calculus for Finance II: Continuous-Time Models.*
- Björk, T. *Arbitrage Theory in Continuous Time.*
- Hull, J. C. *Options, Futures, and Other Derivatives.*

### PDE and numerical methods

- Wilmott, P. *Paul Wilmott on Quantitative Finance.*
- Tavella, D. and Randall, C. *Pricing Financial Instruments: The Finite Difference Method.*
- Andersen, L. and Piterbarg, V. *Interest Rate Modeling.*

### Research extensions

- Dupire local volatility.
- Heston stochastic volatility.
- Deep BSDE methods for high-dimensional PDEs.
- Physics-informed neural networks for option pricing PDEs.
- Rough volatility and fractional Black-Scholes models.
- XVA and nonlinear pricing PDEs/BSDEs.